In [ ]:
import os
import glob
import copy

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from cmocean import cm

In [ ]:
mesh_mask_file = "/ocean/dtaneja/MOAD/analysis-dishika/grid/mesh_mask202108.nc"
original_file = "/ocean/dtaneja/MOAD/analysis-dishika/notebooks/data/hrdps_nemo_3h/HRDPS_NEMO_2008_tair_3h.nc"
recon_dir = "/ocean/dtaneja/MOAD/analysis-dishika/notebooks/results/daily_downscaled_2008_padded"
plot_output_dir = "/ocean/dtaneja/MOAD/analysis-dishika/notebooks/results/monthly_difference_plots_2008"

os.makedirs(plot_output_dir, exist_ok=True)

In [ ]:
with xr.open_dataset(mesh_mask_file) as ds_mesh:
    water_mask = (
        ds_mesh["tmask"]
        .isel(t=0, z=0)
        .load()
        .values
        .astype(bool)
    )

    nemo_lat = ds_mesh["nav_lat"].load().values
    nemo_lon = ds_mesh["nav_lon"].load().values

nemo_y_size, nemo_x_size = water_mask.shape

print("NEMO grid shape:", water_mask.shape)
print("Number of water cells:", water_mask.sum())

In [ ]:
water_mask_da = xr.DataArray(
    water_mask,
    dims=("y", "x"),
    coords={
        "y": np.arange(nemo_y_size),
        "x": np.arange(nemo_x_size),
    },
    name="water_mask",
)

In [ ]:
plt.figure(figsize=(6, 9))

water_mask_da.plot(
    cmap="Blues",
    add_colorbar=False
)

plt.title("NEMO Surface Water Mask")
plt.tight_layout()
plt.show()

In [ ]:
if not os.path.exists(original_file):
    raise FileNotFoundError(
        f"Original HRDPS file not found:\n{original_file}"
    )

ds_original_water = xr.open_dataset(original_file)

print(ds_original_water)

In [ ]:
year_mask = (
    ds_original_water["time_counter"].dt.year == 2008
)

ds_original_2008_water = (
    ds_original_water
    .sel(time_counter=year_mask)
    .sortby("time_counter")
    .load()
)

print("Original 2008 dataset:")
print(ds_original_2008_water)

print(
    "Original number of timestamps:",
    ds_original_2008_water.sizes["time_counter"]
)

print(
    "Original time range:",
    ds_original_2008_water.time_counter.values[0],
    "to",
    ds_original_2008_water.time_counter.values[-1]
)

In [ ]:
if "nemo_j" not in ds_original_2008_water.coords:
    raise KeyError(
        "The original dataset does not contain the nemo_j coordinate."
    )

if "nemo_i" not in ds_original_2008_water.coords:
    raise KeyError(
        "The original dataset does not contain the nemo_i coordinate."
    )

nemo_j_indices = (
    ds_original_2008_water["nemo_j"]
    .values
    .astype(np.int64)
)

nemo_i_indices = (
    ds_original_2008_water["nemo_i"]
    .values
    .astype(np.int64)
)

n_original_times = (
    ds_original_2008_water.sizes["time_counter"]
)

In [ ]:
original_tair_full = np.full(
    (
        n_original_times,
        nemo_y_size,
        nemo_x_size,
    ),
    np.nan,
    dtype=np.float32,
)

In [ ]:

original_tair_full[
    :,
    nemo_j_indices,
    nemo_i_indices,
] = ds_original_2008_water["tair"].values

In [ ]:
atmos_3h = xr.Dataset(
    data_vars={
        "tair": (
            ("time_counter", "y", "x"),
            original_tair_full,
        )
    },
    coords={
        "time_counter": (
            ds_original_2008_water[
                "time_counter"
            ].values
        ),
        "y": np.arange(nemo_y_size),
        "x": np.arange(nemo_x_size),
        "nav_lat": (
            ("y", "x"),
            nemo_lat,
        ),
        "nav_lon": (
            ("y", "x"),
            nemo_lon,
        ),
    },
)

atmos_3h["tair"].attrs = (
    ds_original_2008_water["tair"].attrs.copy()
)

atmos_3h["tair"].attrs["long_name"] = (
    atmos_3h["tair"].attrs.get(
        "long_name",
        "tairitation",
    )
)

atmos_3h.attrs["description"] = (
    "Original HRDPS/GEMLAM tairitation interpolated "
    "onto the NEMO grid and resampled to 3-hourly means."
)

In [ ]:
atmos_3h["tair"] = atmos_3h["tair"].where(
    water_mask_da
)

In [ ]:
print("atmos_3h:")
print(atmos_3h)

print(
    "atmos_3h tair shape:",
    atmos_3h["tair"].shape
)

In [ ]:
recon_files = sorted(
    glob.glob(
        os.path.join(
            recon_dir,
            "*y2008m??d??*.nc",
        )
    )
)

if len(recon_files) == 0:
    raise FileNotFoundError(
        "No 2008 reconstructed files were found in:\n"
        f"{recon_dir}"
    )

print(
    "Number of reconstructed files:",
    len(recon_files)
)

print("First file:", recon_files[0])
print("Last file: ", recon_files[-1])

In [ ]:
try:
    atmos_recon_padded = xr.open_mfdataset(
        recon_files,
        combine="by_coords",
    )

except ValueError:
    print(
        "combine='by_coords' failed. "
        "Using nested concatenation."
    )

    atmos_recon_padded = xr.open_mfdataset(
        recon_files,
        combine="nested",
        concat_dim="time_counter",
    )

In [ ]:
if "time_counter" not in atmos_recon_padded.dims:
    possible_time_dims = [
        dim
        for dim in atmos_recon_padded.dims
        if "time" in dim.lower()
    ]

    if len(possible_time_dims) != 1:
        raise ValueError(
            "Could not uniquely identify the time dimension. "
            f"Dataset dimensions are: {atmos_recon_padded.dims}"
        )

    atmos_recon_padded = atmos_recon_padded.rename(
        {
            possible_time_dims[0]: "time_counter"
        }
    )

In [ ]:
atmos_recon_padded = atmos_recon_padded.sortby(
    "time_counter"
)

recon_time_index = pd.Index(
    pd.to_datetime(
        atmos_recon_padded[
            "time_counter"
        ].values
    )
)

if recon_time_index.has_duplicates:
    keep_times = ~recon_time_index.duplicated()

    atmos_recon_padded = (
        atmos_recon_padded.isel(
            time_counter=keep_times
        )
    )

In [ ]:
recon_2008_mask = (
    atmos_recon_padded[
        "time_counter"
    ].dt.year == 2008
)

atmos_recon_padded = (
    atmos_recon_padded
    .sel(time_counter=recon_2008_mask)
    .sortby("time_counter")
)

In [ ]:
print("Padded reconstructed dataset:")
print(atmos_recon_padded)

print(
    "Variables:",
    list(atmos_recon_padded.data_vars)
)

print(
    "Number of reconstructed timestamps:",
    atmos_recon_padded.sizes["time_counter"]
)

In [ ]:
if "tair" not in atmos_recon_padded:
    raise KeyError(
        "The reconstructed dataset does not contain 'tair'. "
        f"Available variables are: "
        f"{list(atmos_recon_padded.data_vars)}"
    )

In [ ]:
recon_spatial_dims = [
    dim
    for dim in atmos_recon_padded["tair"].dims
    if dim != "time_counter"
]

if len(recon_spatial_dims) != 2:
    raise ValueError(
        "Expected two spatial dimensions for reconstructed tair, "
        f"but found: {recon_spatial_dims}"
    )

recon_y_dim, recon_x_dim = recon_spatial_dims

print("Reconstructed y dimension:", recon_y_dim)
print("Reconstructed x dimension:", recon_x_dim)

print(
    "Padded reconstructed shape:",
    atmos_recon_padded.sizes[recon_y_dim],
    atmos_recon_padded.sizes[recon_x_dim],
)

In [ ]:
recon_y_size = atmos_recon_padded.sizes[
    recon_y_dim
]

recon_x_size = atmos_recon_padded.sizes[
    recon_x_dim
]

extra_y = recon_y_size - nemo_y_size
extra_x = recon_x_size - nemo_x_size

if extra_y < 0 or extra_x < 0:
    raise ValueError(
        "The reconstructed grid is smaller than the original "
        "NEMO grid."
    )

if extra_y % 2 != 0 or extra_x % 2 != 0:
    raise ValueError(
        "The padding is not symmetric. "
        f"Extra y cells: {extra_y}; "
        f"extra x cells: {extra_x}"
    )

pad_y = extra_y // 2
pad_x = extra_x // 2

print("Padding on each y side:", pad_y)

print("Padding on each x side:", pad_x)

In [ ]:
atmos_recon = atmos_recon_padded.isel(
    {
        recon_y_dim: slice(
            pad_y,
            pad_y + nemo_y_size,
        ),
        recon_x_dim: slice(
            pad_x,
            pad_x + nemo_x_size,
        ),
    }
)

In [ ]:
rename_dimensions = {}

if recon_y_dim != "y":
    rename_dimensions[recon_y_dim] = "y"

if recon_x_dim != "x":
    rename_dimensions[recon_x_dim] = "x"

if rename_dimensions:
    atmos_recon = atmos_recon.rename(
        rename_dimensions
    )

In [ ]:
atmos_recon = atmos_recon.assign_coords(
    y=np.arange(nemo_y_size),
    x=np.arange(nemo_x_size),
    nav_lat=(
        ("y", "x"),
        nemo_lat,
    ),
    nav_lon=(
        ("y", "x"),
        nemo_lon,
    ),
)

In [ ]:
atmos_recon = atmos_recon.where(
    water_mask_da
)

In [ ]:
print("atmos_recon:")
print(atmos_recon)

print(
    "Cropped reconstructed tair shape:",
    atmos_recon["tair"].shape
)

In [ ]:
# Original HRDPS/GEMLAM on the NEMO grid
print(atmos_3h)

# Padded downscaled files
print(atmos_recon_padded)

# Downscaled files cropped to the original NEMO grid
print(atmos_recon)

In [ ]:
print(
    "Original timestamps:",
    atmos_3h.sizes["time_counter"]
)

print(
    "Reconstructed timestamps:",
    atmos_recon.sizes["time_counter"]
)

print(
    "Original range:",
    atmos_3h.time_counter.values[0],
    "to",
    atmos_3h.time_counter.values[-1]
)

print(
    "Reconstructed range:",
    atmos_recon.time_counter.values[0],
    "to",
    atmos_recon.time_counter.values[-1]
)

In [ ]:
original_times = pd.to_datetime(
    atmos_3h["time_counter"].values
)

reconstructed_times = pd.to_datetime(
    atmos_recon["time_counter"].values
)

common_times = np.intersect1d(
    original_times.values,
    reconstructed_times.values,
)

if len(common_times) == 0:
    raise ValueError(
        "No matching timestamps were found between the "
        "original and reconstructed datasets."
    )

atmos_3h = atmos_3h.sel(
    time_counter=common_times
)

atmos_recon = atmos_recon.sel(
    time_counter=common_times
)

print(
    "Number of matching timestamps:",
    len(common_times)
)

print(
    "Aligned original shape:",
    atmos_3h["tair"].shape
)

print(
    "Aligned reconstruction shape:",
    atmos_recon["tair"].shape
)

In [ ]:
import calendar


def plot_monthly_differences(
    original,
    reconstructed,
    var,
    dv=None,
):
    # ----------------------------------------------------------
    # Difference at every matching 3-hour timestamp
    # ----------------------------------------------------------

    difference = (
        reconstructed[var]
        - original[var]
    )

    # ----------------------------------------------------------
    # Mean difference for each month
    # ----------------------------------------------------------

    monthly_difference = (
        difference
        .groupby("time_counter.month")
        .mean(
            dim="time_counter",
            skipna=True,
        )
    )

    # Check that all 12 months are present
    available_months = set(
        monthly_difference["month"].values.tolist()
    )

    missing_months = (
        set(range(1, 13))
        - available_months
    )

    if missing_months:
        raise ValueError(
            "Missing reconstructed/original data for months: "
            f"{sorted(missing_months)}"
        )

    # ----------------------------------------------------------
    # One symmetric colour scale for ALL 12 panels
    # ----------------------------------------------------------

    if dv is None:
        valid_values = monthly_difference.values[
            np.isfinite(monthly_difference.values)
        ]

        dv = np.nanpercentile(
            np.abs(valid_values),
            99,
        )

        if dv == 0:
            dv = 1

    difference_cmap = copy.copy(cm.balance)

    difference_cmap = (
        difference_cmap.with_extremes(
            bad="burlywood"
        )
    )

    # ----------------------------------------------------------
    # Create 12 panels
    # ----------------------------------------------------------

    fig, axs = plt.subplots(
        4,
        3,
        figsize=(12, 18),
        constrained_layout=True,
    )

    axs = axs.ravel()

    grid_aspect = (
        monthly_difference.sizes["y"]
        / monthly_difference.sizes["x"]
    )

    for month in range(1, 13):

        ax = axs[month - 1]

        month_field = monthly_difference.sel(
            month=month
        )

        im = month_field.plot(
            ax=ax,
            cmap=difference_cmap,
            vmin=-dv,
            vmax=dv,
            add_colorbar=False,
            add_labels=False,
        )

        ax.set_title(
            calendar.month_name[month],
            fontsize=12,
        )

        ax.set_xlabel("NEMO x index")
        ax.set_ylabel("NEMO y index")

        ax.set_box_aspect(
            grid_aspect
        )

    # ----------------------------------------------------------
    # Shared colour bar
    # ----------------------------------------------------------

    units = original[var].attrs.get(
        "units",
        ""
    )

    cbar = fig.colorbar(
        im,
        ax=axs.tolist(),
        orientation="vertical",
        shrink=0.75,
        pad=0.02,
    )

    if units:
        cbar.set_label(
            f"Reconstruction − Original ({units})"
        )
    else:
        cbar.set_label(
            "Reconstruction − Original"
        )

    return fig, axs, monthly_difference

In [ ]:
fig, axs, monthly_tair_difference = (
    plot_monthly_differences(
        original=atmos_3h,
        reconstructed=atmos_recon,
        var="tair",
    )
)

fig.suptitle(
    "Monthly Mean Difference: Reconstruction − Original GEMLAM, 2008",
    fontsize=16,
)

plt.show()

In [ ]:
output_plot = os.path.join(
    plot_output_dir,
    "tair_monthly_reconstruction_difference_2008.png",
)

fig.savefig(
    output_plot,
    dpi=250,
    bbox_inches="tight",
)

plt.close(fig)

print("Saved:", output_plot)

# Wind plots

In [ ]:
year = 2008

mesh_mask_file = (
    "/ocean/dtaneja/MOAD/analysis-dishika/grid/"
    "mesh_mask202108.nc"
)

original_file = (
    "/ocean/dtaneja/MOAD/analysis-dishika/notebooks/data/"
    "hrdps_nemo_wind_3h/"
    f"HRDPS_NEMO_{year}_wind_3h.nc"
)

recon_dir = (
    "/ocean/dtaneja/MOAD/analysis-dishika/notebooks/results/"
    "daily_downscaled_2008_final"
)

plot_output_dir = (
    "/ocean/dtaneja/MOAD/analysis-dishika/notebooks/results/"
    "monthly_wind_difference_plots_2008"
)

os.makedirs(
    plot_output_dir,
    exist_ok=True,
)

print("Original:", original_file)
print("Reconstruction directory:", recon_dir)

In [ ]:
with xr.open_dataset(mesh_mask_file) as ds_mesh:

    water_mask = (
        ds_mesh["tmask"]
        .isel(t=0, z=0)
        .load()
        .values
        .astype(bool)
    )

    nemo_lat = (
        ds_mesh["nav_lat"]
        .load()
        .values
    )

    nemo_lon = (
        ds_mesh["nav_lon"]
        .load()
        .values
    )


nemo_y_size, nemo_x_size = (
    water_mask.shape
)

print(
    "NEMO grid shape:",
    water_mask.shape,
)

print(
    "Number of surface water cells:",
    water_mask.sum(),
)

In [ ]:
if not os.path.exists(original_file):

    raise FileNotFoundError(
        f"Original wind file not found:\n"
        f"{original_file}"
    )


ds_original_wind = (
    xr.open_dataset(original_file)
    .sortby("time_counter")
)


required_original_variables = [
    "u_wind",
    "v_wind",
    "nemo_j",
    "nemo_i",
]


missing_original_variables = [
    variable
    for variable in required_original_variables
    if variable not in ds_original_wind
    and variable not in ds_original_wind.coords
]


if missing_original_variables:

    raise KeyError(
        "Missing variables/coordinates from "
        f"original wind file: "
        f"{missing_original_variables}"
    )


print(ds_original_wind)

print(
    "Original time range:",
    ds_original_wind.time_counter.values[0],
    "to",
    ds_original_wind.time_counter.values[-1],
)

print(
    "Original U shape:",
    ds_original_wind["u_wind"].shape,
)

print(
    "Original V shape:",
    ds_original_wind["v_wind"].shape,
)

In [ ]:
recon_files = sorted(
    glob.glob(
        os.path.join(
            recon_dir,
            f"*y{year}m??d??*.nc",
        )
    )
)


if len(recon_files) == 0:

    raise FileNotFoundError(
        "No reconstructed files found in:\n"
        f"{recon_dir}"
    )


print(
    "Number of reconstructed files:",
    len(recon_files),
)

print(
    "First reconstructed file:",
    recon_files[0],
)

print(
    "Last reconstructed file:",
    recon_files[-1],
)

In [ ]:
try:

    atmos_recon_padded = xr.open_mfdataset(
        recon_files,
        combine="by_coords",
    )

except ValueError:

    print(
        "combine='by_coords' failed. "
        "Using nested concatenation."
    )

    atmos_recon_padded = xr.open_mfdataset(
        recon_files,
        combine="nested",
        concat_dim="time_counter",
    )


# ----------------------------------------------------------
# Make sure time dimension is called time_counter
# ----------------------------------------------------------

if "time_counter" not in atmos_recon_padded.dims:

    possible_time_dims = [
        dim
        for dim in atmos_recon_padded.dims
        if "time" in dim.lower()
    ]

    if len(possible_time_dims) != 1:

        raise ValueError(
            "Could not identify time dimension. "
            f"Dimensions are: "
            f"{atmos_recon_padded.dims}"
        )

    atmos_recon_padded = (
        atmos_recon_padded.rename(
            {
                possible_time_dims[0]:
                "time_counter"
            }
        )
    )


# ----------------------------------------------------------
# Sort timestamps
# ----------------------------------------------------------

atmos_recon_padded = (
    atmos_recon_padded
    .sortby("time_counter")
)


# ----------------------------------------------------------
# Remove duplicate timestamps
# ----------------------------------------------------------

recon_time_index = pd.Index(
    pd.to_datetime(
        atmos_recon_padded[
            "time_counter"
        ].values
    )
)


if recon_time_index.has_duplicates:

    keep_times = (
        ~recon_time_index.duplicated()
    )

    atmos_recon_padded = (
        atmos_recon_padded.isel(
            time_counter=keep_times
        )
    )


# ----------------------------------------------------------
# Keep only 2008
# ----------------------------------------------------------

year_mask = (
    atmos_recon_padded[
        "time_counter"
    ].dt.year
    == year
)


atmos_recon_padded = (
    atmos_recon_padded
    .sel(time_counter=year_mask)
    .sortby("time_counter")
)


print(atmos_recon_padded)

print(
    "Variables:",
    list(atmos_recon_padded.data_vars),
)

print(
    "Reconstruction time range:",
    atmos_recon_padded.time_counter.values[0],
    "to",
    atmos_recon_padded.time_counter.values[-1],
)

In [ ]:
required_wind_variables = [
    "u_wind",
    "v_wind",
]


missing_wind_variables = [
    variable
    for variable in required_wind_variables
    if variable not in atmos_recon_padded
]


if missing_wind_variables:

    raise KeyError(
        "Missing reconstructed wind variables: "
        f"{missing_wind_variables}\n"
        "Available variables are: "
        f"{list(atmos_recon_padded.data_vars)}"
    )


# ----------------------------------------------------------
# Identify reconstructed spatial dimensions
# ----------------------------------------------------------

recon_spatial_dims = [
    dim
    for dim in atmos_recon_padded[
        "u_wind"
    ].dims
    if dim != "time_counter"
]


if len(recon_spatial_dims) != 2:

    raise ValueError(
        "Expected two spatial dimensions for "
        "u_wind, but found: "
        f"{recon_spatial_dims}"
    )


recon_y_dim, recon_x_dim = (
    recon_spatial_dims
)


print(
    "Reconstructed spatial dimensions:",
    recon_y_dim,
    recon_x_dim,
)


# ----------------------------------------------------------
# Work out padding
# ----------------------------------------------------------

recon_y_size = (
    atmos_recon_padded.sizes[
        recon_y_dim
    ]
)

recon_x_size = (
    atmos_recon_padded.sizes[
        recon_x_dim
    ]
)


extra_y = (
    recon_y_size
    - nemo_y_size
)

extra_x = (
    recon_x_size
    - nemo_x_size
)


if extra_y < 0 or extra_x < 0:

    raise ValueError(
        "Reconstructed grid is smaller "
        "than the NEMO grid."
    )


if (
    extra_y % 2 != 0
    or extra_x % 2 != 0
):

    raise ValueError(
        "Padding is not symmetric. "
        f"Extra y = {extra_y}, "
        f"extra x = {extra_x}"
    )


pad_y = extra_y // 2
pad_x = extra_x // 2


print(
    "Padding on each y side:",
    pad_y,
)

print(
    "Padding on each x side:",
    pad_x,
)


# ----------------------------------------------------------
# Crop to original NEMO dimensions
# ----------------------------------------------------------

atmos_recon = (
    atmos_recon_padded.isel(
        {
            recon_y_dim:
                slice(
                    pad_y,
                    pad_y
                    + nemo_y_size,
                ),

            recon_x_dim:
                slice(
                    pad_x,
                    pad_x
                    + nemo_x_size,
                ),
        }
    )
)


# ----------------------------------------------------------
# Rename spatial dimensions to y and x
# ----------------------------------------------------------

rename_dimensions = {}


if recon_y_dim != "y":

    rename_dimensions[
        recon_y_dim
    ] = "y"


if recon_x_dim != "x":

    rename_dimensions[
        recon_x_dim
    ] = "x"


if rename_dimensions:

    atmos_recon = (
        atmos_recon.rename(
            rename_dimensions
        )
    )


# ----------------------------------------------------------
# Assign NEMO coordinates
# ----------------------------------------------------------

atmos_recon = (
    atmos_recon.assign_coords(
        y=np.arange(
            nemo_y_size
        ),

        x=np.arange(
            nemo_x_size
        ),

        nav_lat=(
            ("y", "x"),
            nemo_lat,
        ),

        nav_lon=(
            ("y", "x"),
            nemo_lon,
        ),
    )
)


print(atmos_recon)

print(
    "Reconstructed U shape:",
    atmos_recon["u_wind"].shape,
)

print(
    "Reconstructed V shape:",
    atmos_recon["v_wind"].shape,
)

In [ ]:
water_j = xr.DataArray(
    ds_original_wind[
        "nemo_j"
    ].values.astype(np.int64),
    dims="water_cell",
)

water_i = xr.DataArray(
    ds_original_wind[
        "nemo_i"
    ].values.astype(np.int64),
    dims="water_cell",
)


recon_u_water = (
    atmos_recon["u_wind"]
    .isel(
        y=water_j,
        x=water_i,
    )
    .reset_coords(
        drop=True
    )
)


recon_v_water = (
    atmos_recon["v_wind"]
    .isel(
        y=water_j,
        x=water_i,
    )
    .reset_coords(
        drop=True
    )
)


recon_wind_water = xr.Dataset(
    {
        "u_wind":
            recon_u_water,

        "v_wind":
            recon_v_water,
    }
)


recon_wind_water = (
    recon_wind_water.assign_coords(
        water_cell=(
            ds_original_wind[
                "water_cell"
            ].values
        )
    )
)


print(recon_wind_water)

print(
    "Reconstructed water-cell U:",
    recon_wind_water[
        "u_wind"
    ].shape,
)

print(
    "Reconstructed water-cell V:",
    recon_wind_water[
        "v_wind"
    ].shape,
)

In [ ]:
original_wind = (
    ds_original_wind[
        [
            "u_wind",
            "v_wind",
        ]
    ]
)


original_wind, recon_wind_water = (
    xr.align(
        original_wind,
        recon_wind_water,
        join="inner",
    )
)


print(
    "Matching timestamps:",
    original_wind.sizes[
        "time_counter"
    ],
)


print(
    "Original aligned U shape:",
    original_wind[
        "u_wind"
    ].shape,
)

print(
    "Reconstructed aligned U shape:",
    recon_wind_water[
        "u_wind"
    ].shape,
)


print(
    "Original aligned V shape:",
    original_wind[
        "v_wind"
    ].shape,
)

print(
    "Reconstructed aligned V shape:",
    recon_wind_water[
        "v_wind"
    ].shape,
)


print(
    "Aligned time range:",
    original_wind.time_counter.values[0],
    "to",
    original_wind.time_counter.values[-1],
)

In [ ]:
original_wind_speed_squared = (
    original_wind[
        "u_wind"
    ] ** 2
    +
    original_wind[
        "v_wind"
    ] ** 2
)


reconstructed_wind_speed_squared = (
    recon_wind_water[
        "u_wind"
    ] ** 2
    +
    recon_wind_water[
        "v_wind"
    ] ** 2
)


wind_speed_squared_difference = (
    reconstructed_wind_speed_squared
    -
    original_wind_speed_squared
)


wind_speed_squared_difference.name = (
    "wind_speed_squared_difference"
)


print(
    wind_speed_squared_difference
)

In [ ]:
monthly_difference_water = (
    wind_speed_squared_difference
    .groupby(
        "time_counter.month"
    )
    .mean(
        dim="time_counter",
        skipna=True,
    )
    .compute()
)


print(
    monthly_difference_water
)

print(
    "Months:",
    monthly_difference_water[
        "month"
    ].values,
)

In [ ]:
available_months = set(
    monthly_difference_water[
        "month"
    ].values.tolist()
)


missing_months = (
    set(range(1, 13))
    - available_months
)


if missing_months:

    raise ValueError(
        "Missing data for months: "
        f"{sorted(missing_months)}"
    )

In [ ]:
monthly_difference_grid = np.full(
    (
        12,
        nemo_y_size,
        nemo_x_size,
    ),
    np.nan,
    dtype=np.float32,
)


nemo_j_indices = (
    ds_original_wind[
        "nemo_j"
    ]
    .values
    .astype(np.int64)
)

nemo_i_indices = (
    ds_original_wind[
        "nemo_i"
    ]
    .values
    .astype(np.int64)
)


for month in range(1, 13):

    monthly_difference_grid[
        month - 1,
        nemo_j_indices,
        nemo_i_indices,
    ] = (
        monthly_difference_water
        .sel(month=month)
        .values
    )


monthly_wind_speed_squared_difference = (
    xr.DataArray(
        monthly_difference_grid,

        dims=(
            "month",
            "y",
            "x",
        ),

        coords={
            "month":
                np.arange(
                    1,
                    13,
                ),

            "y":
                np.arange(
                    nemo_y_size
                ),

            "x":
                np.arange(
                    nemo_x_size
                ),

            "nav_lat": (
                ("y", "x"),
                nemo_lat,
            ),

            "nav_lon": (
                ("y", "x"),
                nemo_lon,
            ),
        },

        name=(
            "wind_speed_squared_difference"
        ),
    )
)


print(
    monthly_wind_speed_squared_difference
)

In [ ]:
valid_values = (
    monthly_wind_speed_squared_difference
    .values
)

valid_values = valid_values[
    np.isfinite(
        valid_values
    )
]


# Same symmetric scale for all 12 panels
dv = np.nanpercentile(
    np.abs(valid_values),
    99,
)


if dv == 0:
    dv = 1


difference_cmap = copy.copy(
    cm.balance
)

difference_cmap = (
    difference_cmap.with_extremes(
        bad="burlywood"
    )
)


fig, axs = plt.subplots(
    4,
    3,
    figsize=(12, 18),
    constrained_layout=True,
)


axs = axs.ravel()


grid_aspect = (
    nemo_y_size
    / nemo_x_size
)


for month in range(1, 13):

    ax = axs[
        month - 1
    ]

    month_field = (
        monthly_wind_speed_squared_difference
        .sel(
            month=month
        )
    )


    im = month_field.plot(
        ax=ax,
        cmap=difference_cmap,
        vmin=-dv,
        vmax=dv,
        add_colorbar=False,
        add_labels=False,
    )


    ax.set_title(
        calendar.month_name[
            month
        ],
        fontsize=12,
    )

    ax.set_xlabel(
        "NEMO x index"
    )

    ax.set_ylabel(
        "NEMO y index"
    )

    ax.set_box_aspect(
        grid_aspect
    )


cbar = fig.colorbar(
    im,
    ax=axs.tolist(),
    orientation="vertical",
    shrink=0.75,
    pad=0.02,
)


cbar.set_label(
    r"Reconstruction − Original Wind Speed$^2$ "
    r"(m$^2$ s$^{-2}$)"
)


fig.suptitle(
    r"Monthly Mean Wind Speed$^2$ Difference: "
    rf"Reconstruction − Original GEMLAM, {year}",
    fontsize=16,
)


plt.show()

In [ ]:
output_plot = os.path.join(
    plot_output_dir,
    (
        "wind_speed_squared_"
        "monthly_difference_"
        f"{year}.png"
    ),
)


fig.savefig(
    output_plot,
    dpi=250,
    bbox_inches="tight",
)


print(
    "Saved:",
    output_plot,
)